In [2]:
pip install selenium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: C:\Users\Pauline\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [101]:
pip install webdriver-manager


[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: C:\Users\Pauline\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [102]:
pip install bs4


[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: C:\Users\Pauline\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import random
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from bs4 import BeautifulSoup
from selenium.webdriver.chrome.options import Options
import csv
import requests
import pandas as pd
import re

from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import os


In [34]:
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Modo headless
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920x1080")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--no-sandbox")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


# CODIGO PARA CREAR WINE_DATA
- Cuidado, que de momento cada vez que se ejecuta, el wine_data.csv aplasta lo de antes, hay que cambiar nombre del csv al final y guardarlo si lo queremos conservar.
- A ver si para despues lo cambio para que las filas se agreguen a lo que ya esta o si es mas comodo tener un csv x tipo de vino. 
- Depende también como proceso los datos, si los junto todo antes en un solo txt o si voy poco a poco

**Falta integrar lo que se hara con Selenium:**
- Caracteristicas
- Nota de sabor 

**Test hecho con 500 blancos:** 7 min, parece que el año no se encuentra siempre pero no supe mejoralo más :( 


In [ ]:
# Leer los enlaces desde un archivo de texto
with open(r'C:\Users\Pauline\Downloads\Proyecto_grupo2_vinos\tintos\def_tintos_merged_150_to_202.txt', 'r', encoding='utf-8-sig') as f:
    urls = f.readlines()  # Lee todos los enlaces en el archivo

# Limitar a los primeros 5 enlaces
#urls = urls[:2]

# Headers para evitar bloqueos
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Inicializar una lista para guardar todos los datos
all_wine_data = []

# Recorrer cada URL en la lista
for url in urls:
    url = url.strip()  # Eliminar cualquier espacio en blanco o salto de línea

    try:
        # Solicitud a la página
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')

        # Inicializar datos
        wine_data = {}

        # País, región, bodega, tipo de vino, uva
        breadcrumbs = soup.find(class_='breadCrumbs__breadCrumbs--2pkcX')
        country = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-country'}).text.strip() if breadcrumbs else 'No disponible'
        region = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-region'}).text.strip() if breadcrumbs else 'No disponible'
        winery = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winery'}).text.strip() if breadcrumbs else 'No disponible'
        wine_type = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winetype'}).text.strip() if breadcrumbs else 'No disponible'
        #grape = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-grape'}).text.strip() if breadcrumbs else 'No disponible'


        #Grape
        grapes = soup.find_all("a", class_="anchor_anchor__m8Qi- wineFacts__link--3aTg9")
        grape = [grape.text.strip() for grape in grapes if "grapes" in grape["href"]]
        grape = ', '.join(grape) if grape else 'No disponible'


        # Nombre del vino 
        wine_headline = soup.find('div', class_='wineHeadline-module__wineHeadline--32Ety')
        if wine_headline:
            # Tomamos todo el texto dentro del bloque, sin intentar separarlo
            name = wine_headline.get_text(strip=True)
        else:
            name = 'No disponible'
        
            # Si el nombre del vino contiene el nombre de la bodega, eliminamos la bodega del nombre
        if winery.lower() in name.lower():
            name = name.replace(winery, '').strip()


        # Año
        button_elements = soup.find_all('button', class_='MuiButtonBase-root')
        year = 'No disponible'

            # Buscar en los botones primero
        for button in button_elements:
            if button.get('aria-label') and button.get('aria-label').isdigit() and len(button.get('aria-label').strip()) == 4:
                year = button.get('aria-label').strip()
                break

            # Si no se encuentra en los botones, buscar en el span con la clase 'vivino-mui-14ngluw-componentChildren'
        if year == 'No disponible':
            year_element = soup.find('span', class_='vivino-mui-14ngluw-componentChildren')
            if year_element and year_element.text.strip().isdigit() and len(year_element.text.strip()) == 4:
                year = year_element.text.strip()

            # Si aún no se ha encontrado, buscar todos los 'span' con la clase 'vintageListRow__year--34Tuc' y tomar el primero
        if year == 'No disponible':
            vintage_section = soup.find('div', id='vintageListSection')
            if vintage_section:
                # Buscar todos los 'span' dentro del div y filtrar aquellos que contienen un año (4 dígitos)
                year_elements = vintage_section.find_all('span', string=re.compile(r'\d{4}'))
                if year_elements:
                    year = year_elements[0].string.strip()  # Usamos .string para obtener solo el texto


        # Precio
        price_element = soup.find(class_='purchaseAvailability__currentPrice--3mO4u')
        if price_element:
            price = price_element.text.replace('€', '').replace('\xa0', '').strip()
        else:
            # Si no lo encuentra, busca el precio en la segunda clase
            price_element = soup.find(class_='purchaseAvailabilityPPC__amount--2_4GT')
            if price_element:
                # Utilizamos regex para encontrar el precio con la coma
                match = re.search(r'\d{1,3}(?:,\d{3})*(?:\.\d+)?', price_element.text)
                if match:
                    price = match.group(0).replace('\xa0', '').strip()
                else:
                    price = 'No disponible'
            else:
                price = 'No disponible'

        
        # Grados de Alcohol
        alcohol_element = soup.find(class_='wineFacts__wineFacts--2Ih8B')
            # Buscar todos los spans dentro de la tabla
        if alcohol_element:
            spans = alcohol_element.find_all('span')
            # Filtrar los spans que contienen un número seguido de '%' (grado de alcohol)
        alcohol = 'No disponible'
        for span in spans:
                # Usamos una expresión regular para buscar un número seguido de '%'
                match = re.search(r'\d+%', span.text.strip())
                if match:
                    alcohol = match.group(0)  # El valor que coincide con la expresión regular
                    alcohol = alcohol.replace('%', '')
                    break  # Detener la búsqueda cuando encontramos el primer grado de alcohol
        else:
            alcohol = 'No disponible'



        # Notas de sabor
        taste_containers = soup.find_all(class_='slider__viewPort--30MrB')
        taste_notes = []
        for container in taste_containers:
            # Encontrar todos los elementos con la clase 'tasteNote__popularKeywords--1gIa2'
            taste_keywords = container.find_all(class_='tasteNote__popularKeywords--1gIa2')

            # Recorrer cada uno de los elementos encontrados y extraer el texto
            for keyword in taste_keywords:
                if keyword.text.strip():  # Solo si no está vacío
                    taste_notes.append(keyword.text.strip())

        # Unir todas las notas en una sola cadena, separada por coma
        taste_notes = ', '.join(taste_notes) if taste_notes else 'No disponible'

        
        # Valoración
        rating_element = soup.find(class_='vivinoRating_averageValue__uDdPM')
        rating = rating_element.text.strip().replace(',', '.') if rating_element else 'No disponible'  # Reemplaza la coma por un punto

        # Maridajes
        food_pairings = soup.find_all(class_='foodPairing__foodImage--2OYHg')
        pairings = [fp['aria-label'] for fp in food_pairings if fp.has_attr('aria-label')]


        # Guardar los datos de esta URL
        wine_data = {
            'Url': url,
            'ID': re.search(r'\/(\d+)$', url).group(1),
            'Nombre': name,
            'Año': year,
            'País': country,
            'Región': region,
            'Bodega': winery,
            'Tipo de vino': wine_type,
            'Uva': grape,
            'Precio': price,
            'Valoración': rating,
            'Contenido de alcohol': alcohol,
            'Maridajes':', '.join(pairings),
            
        }
        all_wine_data.append(wine_data)

    except Exception as e:
        print(f"Error al procesar la URL {url}: {e}")

# Guardar los resultados en un archivo CSV
with open('wine_data.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=all_wine_data[0].keys())
    writer.writeheader()
    writer.writerows(all_wine_data)

# Convertir los datos a un DataFrame y mostrar los primeros registros
df = pd.DataFrame(all_wine_data)
pd.set_option('display.max_colwidth', None)  # No limita el ancho de las celdas
pd.set_option('display.max_rows', None)  # Sin límite de filas
pd.set_option('display.max_columns', None)  # Sin límite de columnas

print(df.head())


Error al procesar la URL Url: Invalid URL 'Url': No scheme supplied. Perhaps you meant https://Url?
Error al procesar la URL https://www.vivino.com/ES/es/rigas-sparkling-medium-dry/w/1157882: 'NoneType' object has no attribute 'text'
Error al procesar la URL https://www.vivino.com/ES/es/skepparps-vingard-grand-prix-solaris-mousserande-brut/w/6175956: 'NoneType' object has no attribute 'text'
Error al procesar la URL https://www.vivino.com/ES/es/divo-brut-veneto-sparkling-v-epuq3/w/3111172: 'NoneType' object has no attribute 'text'
                                                                                 Url  \
0           https://www.vivino.com/ES/es/vinos-sanz-fri-sanz-te-semi-dulce/w/4559173   
1   https://www.vivino.com/ES/es/castellblanc-castellblanch-cava-brut-cava/w/2453836   
2  https://www.vivino.com/ES/es/palacio-de-bornos-verdejo-5-5deg-frizzante/w/2329608   
3   https://www.vivino.com/ES/es/palacio-de-bornos-rosado-5-5deg-frizzante/w/4243274   
4                      

## CODIGOS QUE FALTAN INTEGRAR ##

In [40]:
# URL de la página de Vivino
url = "https://www.vivino.com/ES/es/domaine-raymond-usseglio-and-fils-la-genese/w/9716238"

# Navegar a la página
driver.get(url)

# Mapeo de etiquetas y las clases de las barras de progreso
labels = ["Ligero/Poderoso", "Suave/Tánico", "Seco/Dulce", "Débil/Ácido"]

# Diccionario para guardar los resultados
progress_values = {}

try:
    # Buscar todas las barras de progreso
    progress_elements = driver.find_elements(By.CLASS_NAME, 'indicatorBar__progress--3aXLX')
    
    # Asegurarnos de que tenemos el mismo número de barras de progreso que etiquetas
    if len(progress_elements) == len(labels):
        # Iterar sobre las barras de progreso y asignarles las etiquetas correspondientes
        for i, element in enumerate(progress_elements):
            style = element.get_attribute('style')
            print(f"Barra {i+1}: {style}")  # Imprimir el atributo style para depuración
            # Sacar el valor del left :
            valor_left = re.search(r'left:\s*([0-9.]+)', style).group(1)
            valor_left = round(float(valor_left)/10,1)
            progress_values[labels[i]] = valor_left
        print(progress_values)

            #si el valor de left es 0, no hacemos nada, 
            # si el valor del left no es 0 a 0, tnemos que coger el valor que hay antes del % y /10 y redondearlo para que tenga 1 solo decimal
            #el resultado de eso, hay que asignarlo a cada una de las labels
            
            
    else:
        print("El número de barras de progreso no coincide con el número de etiquetas esperadas.")
    
except Exception as e:
    print(f"Error durante la extracción: {e}")



Barra 1: width: 20%; left: 41.9107%;
Barra 2: width: 20%; left: 59.7054%;
Barra 3: width: 20%; left: 0px;
Barra 4: width: 20%; left: 52.5%;
{'Ligero/Poderoso': 4.2, 'Suave/Tánico': 6.0, 'Seco/Dulce': 0.0, 'Débil/Ácido': 5.2}


In [41]:
#CARACTERISTICAS VINOS
 
# URL de la página de Vivino
url = "https://www.vivino.com/ES/es/domaine-raymond-usseglio-and-fils-la-genese/w/9716238"

# Navegar a la página
driver.get(url)

# Mapeo de etiquetas y las clases de las barras de progreso
labels = ["Ligero/Poderoso", "Suave/Tánico", "Seco/Dulce", "Débil/Ácido"]

# Diccionario para guardar los resultados
progress_values = {}

try:
    # Buscar todas las barras de progreso
    progress_elements = driver.find_elements(By.CLASS_NAME, 'indicatorBar__progress--3aXLX')
    
    # Asegurarnos de que tenemos el mismo número de barras de progreso que etiquetas
    if len(progress_elements) == len(labels):
        # Iterar sobre las barras de progreso y asignarles las etiquetas correspondientes
        for i, element in enumerate(progress_elements):
            style = element.get_attribute('style')
            print(f"Barra {i+1}: {style}")  # Imprimir el atributo style para depuración
            # Sacar el valor del left :
            valor_left = re.search(r'left:\s*([0-9.]+)', style).group(1)
            valor_left = round(float(valor_left)/10,1)
            progress_values[labels[i]] = valor_left
            
    else:
        print("El número de barras de progreso no coincide con el número de etiquetas esperadas.")
    
except Exception as e:
    print(f"Error durante la extracción: {e}")

# Nombre del archivo CSV
csv_filename = "tintos/tintos_test.csv"
# Cargar datos existentes o crear un nuevo DataFrame
if os.path.exists(csv_filename):
    df = pd.read_csv(csv_filename)
else:
    df = pd.DataFrame(columns=["ID"] + labels)
# Convertir ID a string para evitar errores de tipo
df["ID"] = df["ID"].astype(str)
# Verificar si la ID ya existe en el CSV

ID = url.split("/w/")[1].split("?")[0] if "/w/" in url else "Desconocido"

if ID in df["ID"].values:
    df.loc[df["ID"] == ID, labels] = [progress_values[label] for label in labels]
    print(f"Datos actualizados para el vino con ID: {ID}")
else:
    df = pd.concat([df, pd.DataFrame([progress_values])], ignore_index=True)
    print(f"Nuevo registro agregado con ID: {ID}")
# Guardar de nuevo el archivo CSV
df.to_csv(csv_filename, index=False)
print(f"Datos guardados en {csv_filename}")

# Cerrar el navegador
driver.quit()



Barra 1: width: 20%; left: 41.9107%;
Barra 2: width: 20%; left: 59.7054%;
Barra 3: width: 20%; left: 0px;
Barra 4: width: 20%; left: 52.5%;
Datos actualizados para el vino con ID: 9716238
Datos guardados en tintos/tintos_test.csv


In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import os
import time
import re
import pandas as pd

# Set up Chrome options
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Headless mode
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920x1080")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--no-sandbox")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# Initialize the WebDriver
driver = webdriver.Chrome(options=options)

# Read URLs from a .txt file
with open('C:\\Users\\Pauline\\Downloads\\Proyecto_grupo2_vinos\\tintos\\def_tintos_merged_150_to_202.txt', 'r') as file:
    urls_list = [line.strip() for line in file.readlines()]

# Mapeo de etiquetas y las clases de las barras de progreso
labels = ["Ligero/Poderoso", "Suave/Tánico", "Seco/Dulce", "Débil/Ácido"]

# Variables for handling batch processing
batch_size = 100  # Number of URLs per batch
batch_number = 39
start_index = 3800

# Process each URL batch in the list
while start_index < len(urls_list):
    # Slice the URLs for the current batch
    url_batch = urls_list[start_index:start_index + batch_size]
    start_index += batch_size

    # Prepare the DataFrame for this batch
    df = pd.DataFrame(columns=["ID"] + labels)

    # Process each URL in the batch
    for original_url in url_batch:
        print(f"Procesando URL: {original_url}")  # Mostrar progreso
        match = re.search(r'\/(\d+)$', original_url)  # Buscar el ID en la URL
        ID = match.group(1) if match else "Desconocido"  # Si no se encuentra, asignar "Desconocido"

        try:
            # Open the URL
            driver.get(original_url)

            # Wait for the page to load completely
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, 'body')))

            # Buscar todas las barras de progreso
            progress_elements = driver.find_elements(By.CLASS_NAME, 'indicatorBar__progress--3aXLX')
        
            # Asegurarnos de que tenemos el mismo número de barras de progreso que etiquetas
            if len(progress_elements) == len(labels):
                progress_values = {}  # Diccionario para los valores de un vino
                
                # Iterar sobre las barras de progreso y asignarles las etiquetas correspondientes
                for i, element in enumerate(progress_elements):
                    style = element.get_attribute('style')
                    print(f"Barra {i+1}: {style}")  # Imprimir el atributo style para depuración
                    # Sacar el valor del left:
                    valor_left = re.search(r'left:\s*([0-9.]+)', style).group(1)
                    valor_left = round(float(valor_left)/10, 1)  # Ajustar el valor a 1 decimal
                    progress_values[labels[i]] = valor_left

                # Convertir ID a string para evitar errores de tipo
                progress_values["ID"] = ID
                
                # Añadir el registro al DataFrame
                df = pd.concat([df, pd.DataFrame([progress_values])], ignore_index=True)
                print(f"Nuevo registro agregado con ID: {ID}")

            else:
                print("El número de barras de progreso no coincide con el número de etiquetas esperadas.")
            
        except Exception as e:
            print(f"Error durante la extracción: {e}")

    # Guardar los resultados de este lote en un archivo CSV
    batch_filename = f"caracteristicas_tintos_{batch_number}.csv"
    df.to_csv(batch_filename, index=False)
    print(f"Datos guardados en {batch_filename}")

    # Incrementar el número de lote para el siguiente
    batch_number += 1

# Cerrar el navegador
driver.quit()


Procesando URL: https://www.vivino.com/ES/es/baigorri-rioja-b70/w/4120596
Barra 1: width: 20%; left: 54.0062%;
Barra 2: width: 20%; left: 58.8744%;
Barra 3: width: 20%; left: 9.5112%;
Barra 4: width: 20%; left: 61.0279%;
Nuevo registro agregado con ID: 4120596
Procesando URL: https://www.vivino.com/ES/es/fr-doudet-naudin-corton-marechaudes-vieille-vigne-grand-cru/w/2001437


C:\Users\Pauline\AppData\Local\Temp\ipykernel_13788\4174790329.py:76: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([progress_values])], ignore_index=True)


Barra 1: width: 20%; left: 33.6212%;
Barra 2: width: 20%; left: 21.9712%;
Barra 3: width: 20%; left: 0px;
Barra 4: width: 20%; left: 56.4769%;
Nuevo registro agregado con ID: 2001437
Procesando URL: https://www.vivino.com/ES/es/les-parcellaires-de-saulx-les-murgers-nuits-saint-georges-premier-cru/w/7634128
Barra 1: width: 20%; left: 33.9818%;
Barra 2: width: 20%; left: 34.6545%;
Barra 3: width: 20%; left: 0px;
Barra 4: width: 20%; left: 75.2545%;
Nuevo registro agregado con ID: 7634128
Procesando URL: https://www.vivino.com/ES/es/domaine-jacques-prieur-volnay-santenots-1er-cru/w/16559
Barra 1: width: 20%; left: 36.5936%;
Barra 2: width: 20%; left: 20.6789%;
Barra 3: width: 20%; left: 5.57224%;
Barra 4: width: 20%; left: 57.3979%;
Nuevo registro agregado con ID: 16559
Procesando URL: https://www.vivino.com/ES/es/dubreuil-fontaine-corton-clos-du-roi-grand-cru/w/2430081
Barra 1: width: 20%; left: 33.1338%;
Barra 2: width: 20%; left: 20.5177%;
Barra 3: width: 20%; left: 15.9528%;
Barra 4: 

C:\Users\Pauline\AppData\Local\Temp\ipykernel_13788\4174790329.py:76: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([progress_values])], ignore_index=True)


Barra 1: width: 20%; left: 56.6458%;
Barra 2: width: 20%; left: 51.6619%;
Barra 3: width: 20%; left: 10.539%;
Barra 4: width: 20%; left: 58.9169%;
Nuevo registro agregado con ID: 3312739
Procesando URL: https://www.vivino.com/ES/es/celler-batea-sense-red/w/7569250
Barra 1: width: 20%; left: 48.4259%;
Barra 2: width: 20%; left: 39.75%;
Barra 3: width: 20%; left: 23.5075%;
Barra 4: width: 20%; left: 42.4497%;
Nuevo registro agregado con ID: 7569250
Procesando URL: https://www.vivino.com/ES/es/miquel-pons-seventyseven-veremes-negre/w/4030645
Barra 1: width: 20%; left: 63.5027%;
Barra 2: width: 20%; left: 46.2119%;
Barra 3: width: 20%; left: 8.24666%;
Barra 4: width: 20%; left: 45.4164%;
Nuevo registro agregado con ID: 4030645
Procesando URL: https://www.vivino.com/ES/es/bodegas-felix-callejo-flores-de-callejo/w/1387031
Barra 1: width: 20%; left: 58.1659%;
Barra 2: width: 20%; left: 52.6463%;
Barra 3: width: 20%; left: 12.0101%;
Barra 4: width: 20%; left: 56.8804%;
Nuevo registro agregado 

C:\Users\Pauline\AppData\Local\Temp\ipykernel_13788\4174790329.py:76: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([progress_values])], ignore_index=True)


Barra 1: width: 20%; left: 55.6304%;
Barra 2: width: 20%; left: 49.1235%;
Barra 3: width: 20%; left: 1.69613%;
Barra 4: width: 20%; left: 55.5194%;
Nuevo registro agregado con ID: 1176126
Procesando URL: https://www.vivino.com/ES/es/loggia-dei-sani-oro-dei-sani-chianti-chianti/w/7086319
Barra 1: width: 20%; left: 37.5985%;
Barra 2: width: 20%; left: 47.2905%;
Barra 3: width: 20%; left: 15.4225%;
Barra 4: width: 20%; left: 47.6777%;
Nuevo registro agregado con ID: 7086319
Procesando URL: https://www.vivino.com/ES/es/farina-toro-primero/w/2045160
Barra 1: width: 20%; left: 52.3821%;
Barra 2: width: 20%; left: 36.047%;
Barra 3: width: 20%; left: 19.301%;
Barra 4: width: 20%; left: 39.6834%;
Nuevo registro agregado con ID: 2045160
Procesando URL: https://www.vivino.com/ES/es/jp-chenet-reserve-merlot-cabernet/w/1212207
Barra 1: width: 20%; left: 59.5632%;
Barra 2: width: 20%; left: 52.1449%;
Barra 3: width: 20%; left: 7.78113%;
Barra 4: width: 20%; left: 61.3761%;
Nuevo registro agregado co

C:\Users\Pauline\AppData\Local\Temp\ipykernel_13788\4174790329.py:76: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([progress_values])], ignore_index=True)


Barra 1: width: 20%; left: 62.1832%;
Barra 2: width: 20%; left: 50.3276%;
Barra 3: width: 20%; left: 3.93944%;
Barra 4: width: 20%; left: 60.0512%;
Nuevo registro agregado con ID: 9437674
Procesando URL: https://www.vivino.com/ES/es/can-axartell-velomar-tinto/w/8919991
Barra 1: width: 20%; left: 52%;
Barra 2: width: 20%; left: 41%;
Barra 3: width: 20%; left: 18.4%;
Barra 4: width: 20%; left: 39%;
Nuevo registro agregado con ID: 8919991
Procesando URL: https://www.vivino.com/ES/es/vinos-divertidos-camino-a-pardos-garnacha-vinas-viejas-40-old-vines/w/9895044
Barra 1: width: 20%; left: 77.5017%;
Barra 2: width: 20%; left: 43.2483%;
Barra 3: width: 20%; left: 14.1345%;
Barra 4: width: 20%; left: 42.237%;
Nuevo registro agregado con ID: 9895044
Procesando URL: https://www.vivino.com/ES/es/i-tant-garnatxa-negra/w/1481883
Barra 1: width: 20%; left: 63.7427%;
Barra 2: width: 20%; left: 42.4209%;
Barra 3: width: 20%; left: 15.467%;
Barra 4: width: 20%; left: 46.3946%;
Nuevo registro agregado co

KeyboardInterrupt: 

Completo y correcto para vinos tintos:

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import os
import time
import re
import pandas as pd

# Function to save URLs to a file
def save_urls_to_file(urls, batch_number):
    filename = f'caracteristicas_tintos{batch_number}.csv'
    with open(filename, 'w') as file:
        for url in urls:
            file.write(url + '\n')
    print(f"Las nuevas URLs han sido guardadas en '{filename}'.")

# Set up Chrome options
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Headless mode
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920x1080")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--no-sandbox")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# Initialize the WebDriver
driver = webdriver.Chrome(options=options)

# Read URLs from a .txt file
with open('C:\\Users\\Pauline\\Downloads\\Proyecto_grupo2_vinos\\tintos\\def_tintos_merged_150_to_202.txt', 'r') as file:
    urls_list = [line.strip() for line in file.readlines()]


# List to store new URLs
batch_size = 100
batch_number = 1
start_index = 0



# Mapeo de etiquetas y las clases de las barras de progreso
labels = ["Ligero/Poderoso", "Suave/Tánico", "Seco/Dulce", "Débil/Ácido"]


# Process each URL batch in the list
while start_index < len(urls_list):
    # Slice the URLs for the current batch
    url_batch = urls_list[start_index:start_index + batch_size]
    start_index += batch_size

    # Prepare the DataFrame for this batch
    df = pd.DataFrame(columns=["ID"] + labels)



 # Process each URL in the batch
for original_url in url_batch:
    print(f"Procesando URL: {original_url}")  # Mostrar progreso
    match = re.search(r'\/(\d+)$', original_url)  # Buscar el ID en la URL
    ID = match.group(1) if match else "Desconocido"  # Si no se encuentra, asignar "Desconocido"

    try:
        # Open the URL
        driver.get(original_url)

        # Wait for the page to load completely
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, 'body')))

        # Buscar todas las barras de progreso
        progress_elements = driver.find_elements(By.CLASS_NAME, 'indicatorBar__progress--3aXLX')
    
        # Asegurarnos de que tenemos el mismo número de barras de progreso que etiquetas
        if len(progress_elements) == len(labels):
            progress_values = {}  # Diccionario para los valores de un vino
            
            # Iterar sobre las barras de progreso y asignarles las etiquetas correspondientes
            for i, element in enumerate(progress_elements):
                style = element.get_attribute('style')
                print(f"Barra {i+1}: {style}")  # Imprimir el atributo style para depuración
                # Sacar el valor del left:
                valor_left = re.search(r'left:\s*([0-9.]+)', style).group(1)
                valor_left = round(float(valor_left)/10, 1)  # Ajustar el valor a 1 decimal
                progress_values[labels[i]] = valor_left

            # Convertir ID a string para evitar errores de tipo
            progress_values["ID"] = ID
            
            # Añadir el registro al DataFrame
            df = pd.concat([df, pd.DataFrame([progress_values])], ignore_index=True)
            print(f"Nuevo registro agregado con ID: {ID}")

        else:
            print("El número de barras de progreso no coincide con el número de etiquetas esperadas.")
        
    except Exception as e:
        print(f"Error durante la extracción: {e}")

# Guardar los resultados de este lote en un archivo CSV
batch_filename = f"caracteristicas_tintos_{batch_number}.csv"
df.to_csv(batch_filename, index=False)
print(f"Datos guardados en {batch_filename}")

# Incrementar el número de lote para el siguiente
batch_number += 1

# Cerrar el navegador
driver.quit()


Procesando URL: https://www.vivino.com/api/w/1166416
Barra 1: width: 20%; left: 61.9388%;
Barra 2: width: 20%; left: 24.3112%;
Barra 3: width: 20%; left: 2.37755%;
Barra 4: width: 20%; left: 13.2143%;
Nuevo registro agregado con ID: 1166416
Procesando URL: https://www.vivino.com/api/w/1398168


C:\Users\Pauline\AppData\Local\Temp\ipykernel_11856\485000545.py:90: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([progress_values])], ignore_index=True)


Barra 1: width: 20%; left: 69.3891%;
Barra 2: width: 20%; left: 45.2515%;
Barra 3: width: 20%; left: 17.3041%;
Barra 4: width: 20%; left: 35.4189%;
Nuevo registro agregado con ID: 1398168
Procesando URL: https://www.vivino.com/api/w/1151825
Barra 1: width: 20%; left: 72.4142%;
Barra 2: width: 20%; left: 43.4667%;
Barra 3: width: 20%; left: 5.09033%;
Barra 4: width: 20%; left: 47.7417%;
Nuevo registro agregado con ID: 1151825
Procesando URL: https://www.vivino.com/api/w/5317264
Barra 1: width: 20%; left: 30.7576%;
Barra 2: width: 20%; left: 30.7123%;
Barra 3: width: 20%; left: 0.142468%;
Barra 4: width: 20%; left: 38.5344%;
Nuevo registro agregado con ID: 5317264
Procesando URL: https://www.vivino.com/api/w/77989
Barra 1: width: 20%; left: 64.4929%;
Barra 2: width: 20%; left: 42.9563%;
Barra 3: width: 20%; left: 5.41668%;
Barra 4: width: 20%; left: 51.7764%;
Nuevo registro agregado con ID: 77989
Procesando URL: https://www.vivino.com/api/w/1302728
Barra 1: width: 20%; left: 67.8207%;
Ba

In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import os
import time
import re
import pandas as pd



# Set up Chrome options
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Headless mode
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920x1080")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--no-sandbox")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")


# Initialize the WebDriver
driver = webdriver.Chrome(options=options)

# Read URLs from a .txt file
with open('C:\\Users\\Pauline\\Downloads\\Proyecto_grupo2_vinos\\conversor\\def_tintos_txt_150_to_202\\def_tintos150.txt', 'r') as file:
    urls_list = [line.strip() for line in file.readlines()]

urls_list = urls_list[:5]

# Mapeo de etiquetas y las clases de las barras de progreso
labels = ["Ligero/Poderoso", "Suave/Tánico", "Seco/Dulce", "Débil/Ácido"]

# Nombre del archivo CSV
csv_filename = "def_tintos.csv"
# Cargar datos existentes o crear un nuevo DataFrame
if os.path.exists(csv_filename):
    df = pd.read_csv(csv_filename)
else:
    df = pd.DataFrame(columns=["ID"] + labels)


# Diccionario para guardar los resultados
progress_values = {}

# Process each URL in the list
for original_url in urls_list:
    print(f"Procesando URL: {original_url}")  # Show progress
    match = re.search(r'\/(\d+)$', original_url)  # Buscar el ID en la URL
    ID = match.group(1) if match else "Desconocido"  # Si no se encuentra, asignar "Desconocido"

    try:
        # Open the URL
        driver.get(original_url)

        # Wait for the page to load completely (adjust the condition as needed)
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, 'body')))
        #time.sleep(3)

        # Buscar todas las barras de progreso
        progress_elements = driver.find_elements(By.CLASS_NAME, 'indicatorBar__progress--3aXLX')
    
        # Asegurarnos de que tenemos el mismo número de barras de progreso que etiquetas
        if len(progress_elements) == len(labels):
            # Iterar sobre las barras de progreso y asignarles las etiquetas correspondientes
            for i, element in enumerate(progress_elements):
                style = element.get_attribute('style')
                print(f"Barra {i+1}: {style}")  # Imprimir el atributo style para depuración
                # Sacar el valor del left :
                valor_left = re.search(r'left:\s*([0-9.]+)', style).group(1)
                valor_left = round(float(valor_left)/10,1)
                progress_values[labels[i]] = valor_left
                
    
                # Convertir ID a string para evitar errores de tipo
                df["ID"] = df["ID"].astype(str)
                if ID in df["ID"].values:
                    df.loc[df["ID"] == ID, labels] = [progress_values[label] for label in labels]
                    print(f"Datos actualizados para el vino con ID: {ID}")
                else:
                    df = pd.concat([df, pd.DataFrame([progress_values])], ignore_index=True)
                    print(f"Nuevo registro agregado con ID: {ID}")

        else:
            print("El número de barras de progreso no coincide con el número de etiquetas esperadas.")
    
    except Exception as e:
        print(f"Error durante la extracción: {e}")

# Guardar de nuevo el archivo CSV
df.to_csv(csv_filename, index=False)
print(f"Datos guardados en {csv_filename}")

# Cerrar el navegador
driver.quit()

Procesando URL: https://www.vivino.com/ES/es/domaine-raymond-usseglio-and-fils-la-genese/w/9716238
Barra 1: width: 20%; left: 41.9107%;
Nuevo registro agregado con ID: 9716238
Barra 2: width: 20%; left: 59.7054%;
Nuevo registro agregado con ID: 9716238
Barra 3: width: 20%; left: 0px;
Nuevo registro agregado con ID: 9716238
Barra 4: width: 20%; left: 52.5%;
Nuevo registro agregado con ID: 9716238
Procesando URL: https://www.vivino.com/ES/es/sandro-fay-valgella-valtellina-superiore/w/1944452
Barra 1: width: 20%; left: 51.9034%;
Nuevo registro agregado con ID: 1944452
Barra 2: width: 20%; left: 73.2076%;
Nuevo registro agregado con ID: 1944452
Barra 3: width: 20%; left: 0px;
Nuevo registro agregado con ID: 1944452
Barra 4: width: 20%; left: 80%;
Nuevo registro agregado con ID: 1944452
Procesando URL: https://www.vivino.com/ES/es/romanin-les-baux-de-provence-rouge/w/1214057
Barra 1: width: 20%; left: 66.2698%;
Nuevo registro agregado con ID: 1214057
Barra 2: width: 20%; left: 56.6894%;
Nue

KeyboardInterrupt: 

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import os
import time



# Set up Chrome options
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Headless mode
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920x1080")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--no-sandbox")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# Initialize the WebDriver
driver = webdriver.Chrome(options=options)

# Read URLs from a .txt file
with open('C:\\Users\\Pauline\\Downloads\\Proyecto_grupo2_vinos\\espumosos\\all_espumosos.txt', 'r') as file:
    urls_list = [line.strip() for line in file.readlines()]

# Mapeo de etiquetas y las clases de las barras de progreso
labels = ["Ligero/Poderoso", "Débil/Ácido", "Amable/Con Burbujas"]

# Diccionario para guardar los resultados
progress_values = {}

# Process each URL in the list
for original_url in urls_list:
    print(f"Procesando URL: {original_url}")  # Show progress

    try:
        # Open the URL
        driver.get(original_url)

        # Wait for the page to load completely (adjust the condition as needed)
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, 'body')))
        #time.sleep(3)

        # Buscar todas las barras de progreso
        progress_elements = driver.find_elements(By.CLASS_NAME, 'indicatorBar__progress--3aXLX')
    
        # Asegurarnos de que tenemos el mismo número de barras de progreso que etiquetas
        if len(progress_elements) == len(labels):
            # Iterar sobre las barras de progreso y asignarles las etiquetas correspondientes
            for i, element in enumerate(progress_elements):
                style = element.get_attribute('style')
                print(f"Barra {i+1}: {style}")  # Imprimir el atributo style para depuración
                # Sacar el valor del left :
                valor_left = re.search(r'left:\s*([0-9.]+)', style).group(1)
                valor_left = round(float(valor_left)/10,1)
                progress_values[labels[i]] = valor_left
            
        else:
            print("El número de barras de progreso no coincide con el número de etiquetas esperadas.")
    
    except Exception as e:
        print(f"Error durante la extracción: {e}")

# Nombre del archivo CSV
csv_filename = "def_tintos.csv"
# Cargar datos existentes o crear un nuevo DataFrame
if os.path.exists(csv_filename):
    df = pd.read_csv(csv_filename)
else:
    df = pd.DataFrame(columns=["ID"] + labels)
# Convertir ID a string para evitar errores de tipo
df["ID"] = df["ID"].astype(str)
# Verificar si la ID ya existe en el CSV

ID = url.split("/w/")[1].split("?")[0] if "/w/" in url else "Desconocido"

if ID in df["ID"].values:
    df.loc[df["ID"] == ID, labels] = [progress_values[label] for label in labels]
    print(f"Datos actualizados para el vino con ID: {ID}")
else:
    df = pd.concat([df, pd.DataFrame([progress_values])], ignore_index=True)
    print(f"Nuevo registro agregado con ID: {ID}")
# Guardar de nuevo el archivo CSV
df.to_csv(csv_filename, index=False)
print(f"Datos guardados en {csv_filename}")

# Cerrar el navegador
driver.quit()

In [10]:
#CARACTERISTICAS ESPUMOSOS:


# URL de la página de Vivino
url = "https://www.vivino.com/ES/es/bollinger-vieilles-vignes-francaises-blanc-de-noirs-brut-champagne/w/18938?year=2009&price_id=30860996"

# Navegar a la página
driver.get(url)

labels = ["Ligero/Poderoso", "Débil/Ácido", "Amable/Con Burbujas"]

# Diccionario para guardar los resultados
progress_values = {}

try:
    # Buscar todas las barras de progreso
    progress_elements = driver.find_elements(By.CLASS_NAME, 'indicatorBar__progress--3aXLX')
    
    # Asegurarnos de que tenemos el mismo número de barras de progreso que etiquetas
    if len(progress_elements) == len(labels):
        # Iterar sobre las barras de progreso y asignarles las etiquetas correspondientes
        for i, element in enumerate(progress_elements):
            left_value = element.get_attribute('style').split('left: ')[1].split('%')[0] if 'left' in element.get_attribute('style') else None
            
            if left_value:
                # Convertir a float y mapearlo con los valores predefinidos
                value = round(float(left_value) / 10, 1)
                # Asignar la etiqueta correspondiente con los valores predefinidos
                progress_values[labels[i]] = value
                print(f"{labels[i]}: {value}")
            else:
                print(f"No se encontró el atributo 'left' para {labels[i]}.")
    else:
        print("El número de barras de progreso no coincide con el número de etiquetas esperadas.")
    
except Exception as e:
    print(f"Error durante la extracción: {e}")

# Cerrar el navegador
driver.quit()


Ligero/Poderoso: 7.0
Débil/Ácido: 7.3
Amable/Con Burbujas: 7.0


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import os
import time
import re
import pandas as pd

# Set up Chrome options
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Headless mode
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920x1080")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--no-sandbox")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# Initialize the WebDriver
driver = webdriver.Chrome(options=options)

# Read URLs from a .txt file
with open('C:\\Users\\Pauline\\Downloads\\Proyecto_grupo2_vinos\\conversor\\def_tintos_txt_150_to_202\\def_tintos150.txt', 'r') as file:
    urls_list = [line.strip() for line in file.readlines()]

urls_list = urls_list[:101]  # Limitar a las primeras 5 URLs

# Mapeo de etiquetas y las clases de las barras de progreso
labels = ["Ligero/Poderoso", "Débil/Ácido", "Amable/Con Burbujas"]

# Nombre del archivo CSV
csv_filename = "caracteristicas_tintos.csv"

# Cargar datos existentes o crear un nuevo DataFrame
if os.path.exists(csv_filename):
    df = pd.read_csv(csv_filename)
else:
    df = pd.DataFrame(columns=["ID"] + labels)

# Process each URL in the list
for original_url in urls_list:
    print(f"Procesando URL: {original_url}")  # Mostrar progreso
    match = re.search(r'\/(\d+)$', original_url)  # Buscar el ID en la URL
    ID = match.group(1) if match else "Desconocido"  # Si no se encuentra, asignar "Desconocido"

    try:
        # Open the URL
        driver.get(original_url)

        # Wait for the page to load completely
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, 'body')))

        # Buscar todas las barras de progreso
        progress_elements = driver.find_elements(By.CLASS_NAME, 'indicatorBar__progress--3aXLX')
    
        # Asegurarnos de que tenemos el mismo número de barras de progreso que etiquetas
        if len(progress_elements) == len(labels):
            progress_values = {}  # Diccionario para los valores de un vino
            
            # Iterar sobre las barras de progreso y asignarles las etiquetas correspondientes
            for i, element in enumerate(progress_elements):
                style = element.get_attribute('style')
                print(f"Barra {i+1}: {style}")  # Imprimir el atributo style para depuración
                # Sacar el valor del left:
                valor_left = re.search(r'left:\s*([0-9.]+)', style).group(1)
                valor_left = round(float(valor_left)/10, 1)  # Ajustar el valor a 1 decimal
                progress_values[labels[i]] = valor_left

            # Convertir ID a string para evitar errores de tipo
            progress_values["ID"] = ID

            # Verificar si ya existe el ID en el DataFrame
            if ID in df["ID"].values:
                df.loc[df["ID"] == ID, labels] = [progress_values[label] for label in labels]
                print(f"Datos actualizados para el vino con ID: {ID}")
            else:
                df = pd.concat([df, pd.DataFrame([progress_values])], ignore_index=True)
                print(f"Nuevo registro agregado con ID: {ID}")

        else:
            print("El número de barras de progreso no coincide con el número de etiquetas esperadas.")
    
    except Exception as e:
        print(f"Error durante la extracción: {e}")

# Guardar el DataFrame actualizado en el archivo CSV
df.to_csv(csv_filename, index=False)
print(f"Datos guardados en {csv_filename}")

# Cerrar el navegador
driver.quit()


Waiting for page to load...
Waiting for taste note cards...
Taste note cards found. Extracting data...
Extracting taste note data...
Taste Note: 
Mentions: 
Background Color: rgba(140, 84, 51, 1)
Extracting taste note data...
Taste Note: 
Mentions: 
Background Color: rgba(44, 62, 128, 1)
Extracting taste note data...
Taste Note: 
Mentions: 
Background Color: rgba(181, 155, 111, 1)
Extracting taste note data...
Taste Note: 
Mentions: 
Background Color: rgba(177, 145, 87, 1)
Extracting taste note data...
Taste Note: 
Mentions: 
Background Color: rgba(199, 58, 49, 1)
Extracting taste note data...
Taste Note: 
Mentions: 
Background Color: rgba(248, 102, 118, 1)
Extracting taste note data...
Taste Note: 
Mentions: 
Background Color: rgba(195, 131, 73, 1)
Extracting taste note data...
Taste Note: 
Mentions: 
Background Color: rgba(185, 101, 93, 1)
Extracting taste note data...
Taste Note: 
Mentions: 
Background Color: rgba(158, 110, 64, 1)
Extracting taste note data...
Taste Note: 
Mentions:

# URL uno por uno
Todo ok salvo nota de sabor


In [71]:
import requests
from bs4 import BeautifulSoup
import csv
import pandas as pd

# URL de la página de Vivino
url = "https://www.vivino.com/ES/es/mestres-coquet-gran-reserva-brut-nature/w/4311421"

# Headers para evitar bloqueos
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Solicitud a la página
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')

# Inicializar datos
wine_data = {}

try:
    # Nombre del vino: Ahora extraemos solo el texto del nombre (después de la bodega)
    wine_headline = soup.find('div', class_='wineHeadline-module__wineHeadline--32Ety')
    if wine_headline:
        # Obtener todo el texto dentro del div, luego buscar solo la parte que es el nombre del vino
        text_parts = wine_headline.get_text(strip=True).split(" ")  # Separar por espacios
        # Todo lo que venga después del primer elemento (que sería la bodega)
        name = " ".join(text_parts[1:])  # Tomamos todo después del primer elemento (la bodega)
    else:
        name = 'No disponible'

    #Año :
    # Buscar todos los botones en la página
    button_elements = soup.find_all('button', class_='MuiButtonBase-root')

    # Inicializar año como 'No disponible'
    year = 'No disponible'

    # Iterar sobre todos los botones y buscar uno que contenga un año (4 dígitos)
    for button in button_elements:
        # Verificamos si el aria-label o el texto del botón contiene un año válido
        if button.get('aria-label') and button.get('aria-label').isdigit() and len(button.get('aria-label').strip()) == 4:
            year = button.get('aria-label').strip()
            break  # Si encontramos el año, terminamos el bucle

    if year == 'No disponible':
    # Si no encontramos el año en los botones, intentamos otra estrategia (como se hizo anteriormente)
        year_element = soup.find('span', class_='vivino-mui-14ngluw-componentChildren')
        if year_element and year_element.text.strip().isdigit() and len(year_element.text.strip()) == 4:
            year = year_element.text.strip()




    # País, región, bodega, tipo de vino, uva
    breadcrumbs = soup.find(class_='breadCrumbs__breadCrumbs--2pkcX')
    country = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-country'}).text.strip() if breadcrumbs else 'No disponible'
    region = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-region'}).text.strip() if breadcrumbs else 'No disponible'
    winery = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winery'}).text.strip() if breadcrumbs else 'No disponible'
    wine_type = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winetype'}).text.strip() if breadcrumbs else 'No disponible'
    grape = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-grape'}).text.strip() if breadcrumbs else 'No disponible'

    # Precio
    price_element = soup.find(class_='purchaseAvailability__currentPrice--3mO4u')
    price = price_element.text.replace('€', '').replace('\xa0', '').strip() if price_element else 'No disponible'

    # Valoración
    rating_element = soup.find(class_='vivinoRating_averageValue__uDdPM')
    rating = rating_element.text.strip().replace(',', '.') if rating_element else 'No disponible'  # Reemplaza la coma por un punto

    # Notas de sabor
    taste_containers = soup.find_all(class_='tasteNote__textContainer--2xPXc')
    taste_notes = []
    for container in taste_containers:
        taste_keywords = container.find(class_='tasteNote__popularKeywords--1gIa2')
        if taste_keywords:
            taste_notes.append(taste_keywords.text.strip())
    taste_notes = ', '.join(taste_notes) if taste_notes else 'No disponible'

    # Maridajes
    food_pairings = soup.find_all(class_='foodPairing__foodImage--2OYHg')
    pairings = [fp['aria-label'] for fp in food_pairings if fp.has_attr('aria-label')]



    # Guardar datos
    wine_data = {
        'Nombre': name,
        'Año': year,
        'País': country,
        'Región': region,
        'Bodega': winery,
        'Tipo de vino': wine_type,
        'Uva': grape,
        'Precio': price,
        'Valoración': rating,
        'Notas de sabor': taste_notes,
        'Maridajes': ', '.join(pairings),
         }

except Exception as e:
    print(f"Error durante la extracción: {e}")

# Guardar en CSV
with open('wine_data.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=wine_data.keys())
    writer.writeheader()
    writer.writerow(wine_data)

# Convertir a DataFrame y mostrar los primeros registros
pd.set_option('display.max_colwidth', None)  # No limita el ancho de las celdas
pd.set_option('display.max_rows', None)   # Sin límite de filas
pd.set_option('display.max_columns', None)  # Sin límite de columnas

df = pd.DataFrame([wine_data])
print(df.head())



                     Nombre   Año    País Región   Bodega   Tipo de vino  \
0  Gran Reserva Brut Nature  2019  España   Cava  Mestres  Vino espumoso   

      Uva Precio Valoración Notas de sabor  \
0  Mezcla  15.87        3.9  No disponible   

                                                                    Maridajes  
0  Marisco, Aperitivos y tentempiés, Pescado blanco, Aperitivo, Carne adobada  


# OTRO CODIGO DE VICENTE
Texto definitivo para coger las url de archivo txt

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import pandas as pd
import time

# Headers para evitar bloqueos
headers = {
    'User -Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Inicializar lista para almacenar todos los datos de vino
all_wine_data = []

# Leer las URLs desde un archivo de texto
with open('def_espumoso200.txt', 'r') as file:
    urls = file.readlines()

# Iterar sobre cada URL
for index, url in enumerate(urls, start=1):
    url = url.strip()  # Eliminar espacios en blanco
    wine_data = {}  # Inicializar datos para cada vino

    print(f"Procesando URL {index}/{len(urls)}: {url}")  # Mostrar el progreso

    try:
        # Solicitud a la página
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')

        # Nombre y año
        wine_headline = soup.find(class_='wineHeadline-module__vintage--1UHSo')
        if wine_headline:
            name = wine_headline.find('a').text.strip() if wine_headline.find('a') else 'No disponible'
            year = wine_headline.text.strip().split()[-1]  # Última palabra debería ser el año
        else:
            name = 'No disponible'
            year = 'No disponible'

        # País, región, bodega, tipo de vino, uva
        breadcrumbs = soup.find(class_='breadCrumbs__breadCrumbs--2pkcX')
        country = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-country'}).text.strip() if breadcrumbs else 'No disponible'
        region = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-region'}).text.strip() if breadcrumbs else 'No disponible'
        winery = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winery'}).text.strip() if breadcrumbs else 'No disponible'
        wine_type = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winetype'}).text.strip() if breadcrumbs else 'No disponible'
        grape = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-grape'}).text.strip() if breadcrumbs else 'No disponible'

        # Precio
        price_element = soup.find(class_='purchaseAvailability__currentPrice--3mO4u')
        price = price_element.text.replace('€', '').replace('\xa0', '').strip() if price_element else 'No disponible'

        # Valoración
        rating_element = soup.find(class_='vivinoRating_averageValue__uDdPM')
        rating = rating_element.text.strip().replace(',', '.') if rating_element else 'No disponible'  # Reemplaza la coma por un punto

        # Notas de sabor
        taste_containers = soup.find_all(class_='tasteNote__textContainer--2xPXc')
        taste_notes = []
        for container in taste_containers:
            taste_keywords = container.find(class_='tasteNote__popularKeywords--1gIa2')
            if taste_keywords:
                taste_notes.append(taste_keywords.text.strip())
        taste_notes = ', '.join(taste_notes) if taste_notes else 'No disponible'

        # Maridajes
        food_pairings = soup.find_all(class_='foodPairing__foodImage--2OYHg')
        pairings = [fp['aria-label'] for fp in food_pairings if fp.has_attr('aria-label')]

        # Guardar datos
        wine_data = {
            'Nombre': name,
            'Año': year,
            'País': country,
            'Región': region,
            'Bodega': winery,
            'Tipo de vino': wine_type,
            'Uva': grape,
            'Precio': price,
            'Valoración': rating,
            'Notas de sabor': taste_notes,
            'Maridajes': ', '.join(pairings),
        }

        all_wine_data.append(wine_data)  # Agregar datos a la lista

    except Exception as e:
        print(f"Error durante la extracción de {url}: {e}")

    # Esperar 2 segundos antes de la siguiente solicitud
    time.sleep(2)

# Guardar todos los datos en un CSV
with open('wine_data.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=all_wine_data[0].keys())
    writer.writeheader()
    writer.writerows(all_wine_data)

# Convertir a DataFrame y mostrar los primeros registros
df = pd.DataFrame(all_wine_data)
print(df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'def_espumoso200.txt'